# Ghi điểm cross-encoder để thử cách trộn rerank (Colab GPU)

Notebook này **chỉ chạy model**: với mỗi query (có dấu và không dấu), ghi thứ tự hybrid top-100 và điểm cross-encoder của top-30 theo đúng thứ tự hybrid. Cấu hình = luồng text hiện tại: e5 bất đối xứng, BM25 `fold=add`, bge-reranker-base, max_length 256.

Việc **chọn** cách trộn (replace / rrf / blend) làm trên máy local bằng `select_rerank_fusion.py`, không cần GPU. Lưới và quy tắc chọn đã chốt trong docstring của script đó.

- dev 500 × 2 bản: vài phút
- test 2.675 × 2 bản: lâu hơn (khoảng 30–45 phút trên T4). Ghi ở đây để khỏi quay lại Colab, nhưng **không đọc** trước khi đã chọn trên dev.

**Dữ liệu sẽ rời máy sang Google Colab.** Văn bản đã scrub nhưng vẫn còn tên service, mã lỗi, luồng nghiệp vụ.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "sentence-transformers==6.0.1"

In [ ]:
from google.colab import files
up = files.upload()            # chon fusion_bundle.zip
BUNDLE = next(iter(up))
!unzip -oq {BUNDLE} -d /content/work
%cd /content/work
!ls pipeline data/processed

## 1. Dev

In [ ]:
!python pipeline/dump_rerank_scores.py --split dev --device cuda --batch-size 64

Kiểm tra nhanh (không phải bước chọn): cột `replace` của `khong_dau` phải có R@10 gần **0,566**, `hybrid` gần **0,534**; `goc` replace gần **0,636**. Ô này chỉ in số, **không** ghi file lựa chọn.

In [ ]:
!python pipeline/select_rerank_fusion.py --split dev --n-boot 500 --choice /tmp/choice_check.json --out /tmp/fusion_check.json | head -20

## 2. Test (chỉ ghi điểm, không phân tích)

In [ ]:
!python pipeline/dump_rerank_scores.py --split test --device cuda --batch-size 64

## 3. Tải về

Giải nén vào `D:/viettel/semantic/` rồi báo lại.

In [ ]:
!zip -q rerank_scores.zip data/processed/rerank_scores_dev.jsonl data/processed/rerank_scores_test.jsonl
!ls -la rerank_scores.zip
files.download("rerank_scores.zip")